# FatesGS – Custom-Data Colab Pipeline (v2)

Sparse-view 3D surface reconstruction based on **2D Gaussian Splatting**.

This notebook covers the full workflow:
1. Environment setup
2. Dataset download & exploration
3. Custom calib → COLMAP conversion
4. Monocular depth estimation (Marigold)
5. Initial point-cloud generation
6. FatesGS training
7. Mesh extraction & visualisation

## 0. Configuration

In [ ]:
# ─── All configurable paths & hyper-parameters ───────────────────────
# Google Drive
USE_DRIVE        = True
DRIVE_BASE       = "/content/drive/MyDrive/FatesGS"

# Repository
REPO_URL         = "https://github.com/BAEJUNHAK/FatesGS.git"
REPO_DIR         = "/content/FatesGS"

# Raw dataset (Google Drive file)
GDRIVE_FILE_ID   = "1dnj1s-mqIuS6OcdSr5CczBr9u8yBzUUB"
RAW_DATA_DIR     = "/content/raw_dataset"

# Converted dataset
DATA_ROOT        = "/content/data"
SCENE_NAME       = "custom_object"
CONVERTED_DIR    = f"{DATA_ROOT}/{SCENE_NAME}"

# View selection
N_TRAIN_VIEWS    = 3
N_SOURCE_PAIRS   = 2

# Training
RESOLUTION       = 1          # 1 → native 800×800
ITERATIONS       = 15000

# Loss weights
LAMBDA_FEAT      = 1.5
LAMBDA_DEPTH     = 10.0
LAMBDA_DIST      = 10000.0
LAMBDA_NORMAL    = 0.05
LAMBDA_DSSIM     = 0.2

# Mesh extraction
MESH_RES         = 1024
NUM_CLUSTER      = 50

# Output
OUTPUT_DIR       = f"{DRIVE_BASE}/output/{SCENE_NAME}"

print("Configuration loaded.")

## 1. Environment Setup

### 1-1. GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device   : {torch.cuda.get_device_name(0)}")

### 1-2. Mount Google Drive

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Drive mounted. Base dir: {DRIVE_BASE}")
else:
    print("Google Drive mounting skipped.")

### 1-3. Clone repo & install Python dependencies

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")

%cd {REPO_DIR}

!pip install -q pillow open3d mediapy lpips scikit-image tqdm \
    trimesh plyfile pymeshlab opencv-python scikit-learn plotly gdown
print("\nPython dependencies installed.")

### 1-4. Build CUDA sub-modules

In [ ]:
%cd {REPO_DIR}

import os, torch

# ── CUDA environment setup ──
print(f"PyTorch: {torch.__version__}")
print(f"CUDA (torch): {torch.version.cuda}")

# Set CUDA_HOME if missing
if "CUDA_HOME" not in os.environ:
    import glob
    cuda_dirs = sorted(glob.glob("/usr/local/cuda*"), reverse=True)
    if cuda_dirs:
        os.environ["CUDA_HOME"] = cuda_dirs[0]
        print(f"Set CUDA_HOME = {cuda_dirs[0]}")

# Fetch glm submodule (required for diff-surfel-rasterization)
glm_dir = f"{REPO_DIR}/submodules/diff-surfel-rasterization/third_party/glm"
if not os.path.exists(os.path.join(glm_dir, "glm", "glm.hpp")):
    !git submodule update --init --recursive 2>/dev/null || true
    if not os.path.exists(os.path.join(glm_dir, "glm", "glm.hpp")):
        !git clone https://github.com/g-truc/glm.git {glm_dir}
    print("glm library ready.")
else:
    print("glm already exists.")

# Build submodules
print("\nBuilding diff-surfel-rasterization ...")
!pip install submodules/diff-surfel-rasterization 2>&1 | tail -5

print("\nBuilding simple-knn ...")
!pip install submodules/simple-knn 2>&1 | tail -5

# Verify
print()
try:
    from diff_surfel_rasterization import GaussianRasterizer
    print("[OK] diff-surfel-rasterization")
except ImportError as e:
    print(f"[FAIL] diff-surfel-rasterization: {e}")

try:
    from simple_knn._C import distCUDA2
    print("[OK] simple-knn")
except ImportError as e:
    print(f"[FAIL] simple-knn: {e}")

### 1-5. Install PyTorch3D

In [ ]:
import torch, sys

pyt_version_str = torch.__version__.split("+")[0].replace(".", "")
cuda_version_str = torch.version.cuda.replace(".", "")
wheel_url = (
    f"https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/"
    f"py3{sys.version_info.minor}_cu{cuda_version_str}_pyt{pyt_version_str}/"
    f"pytorch3d-0.7.8-cp3{sys.version_info.minor}-cp3{sys.version_info.minor}"
    f"-linux_x86_64.whl"
)
print(f"Trying pre-built wheel:\n  {wheel_url}")
import subprocess, os
ret = subprocess.run(
    ["pip", "install", "--no-index", "--no-deps", wheel_url],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("Pre-built wheel not found – building from source (takes ~8 min) ...")
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"
else:
    print("PyTorch3D installed from wheel.")

### 1-6. Verify all imports

In [ ]:
import importlib, sys
required = [
    "torch", "torchvision", "numpy", "PIL", "cv2",
    "open3d", "trimesh", "plyfile", "sklearn",
    "mediapy", "lpips", "plotly", "gdown", "tqdm",
]
ok = True
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"MISSING: {mod}")
        ok = False
if ok:
    print("All required packages available.")

## 2. Download & Explore Raw Dataset

### 2-1. Download dataset.zip from Google Drive

In [ ]:
import os, gdown, zipfile

zip_path = "/content/dataset.zip"
if not os.path.exists(zip_path):
    gdown.download(id=GDRIVE_FILE_ID, output=zip_path, quiet=False)
    print(f"Downloaded → {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
else:
    print(f"Already downloaded: {zip_path}")

### 2-2. Unzip & explore

In [ ]:
import os, zipfile, glob

os.makedirs(RAW_DATA_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(RAW_DATA_DIR)
print(f"Extracted to {RAW_DATA_DIR}")

# Auto-find the directory containing calib_0000.ini
data_dir = None
for root, dirs, files in os.walk(RAW_DATA_DIR):
    if "calib_0000.ini" in files:
        data_dir = root
        break
assert data_dir is not None, "Could not find calib_0000.ini in extracted data!"
print(f"Data directory: {data_dir}")

rgb_files   = sorted(glob.glob(os.path.join(data_dir, "rgb_*.png")))
calib_files = sorted(glob.glob(os.path.join(data_dir, "calib_*.ini")))
print(f"RGB images : {len(rgb_files)}")
print(f"Calib files: {len(calib_files)}")

# Show sample calib
print("\n--- Sample calib (calib_0000.ini) ---")
with open(os.path.join(data_dir, "calib_0000.ini")) as f:
    print(f.read()[:600])

### 2-3. Preview sample images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np, random

n_show = min(8, len(rgb_files))
indices = sorted(random.sample(range(len(rgb_files)), n_show))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, idx in zip(axes.flat, indices):
    img = Image.open(rgb_files[idx])
    ax.imshow(np.array(img))
    ax.set_title(os.path.basename(rgb_files[idx]), fontsize=9)
    ax.axis("off")
plt.suptitle(f"Sample images ({len(rgb_files)} total)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Data Format Conversion (Custom → COLMAP)

### 3-1. Parse calibration files

In [ ]:
import configparser, re, numpy as np, os, glob

def parse_matrix(s, rows, cols):
    # Parse 'Matrix:R:C:v1,v2,...' or just the comma-separated values.
    if s.startswith("Matrix"):
        parts = s.split(":")
        vals = parts[3].split(",")
    else:
        vals = s.split(",")
    return np.array([float(v) for v in vals]).reshape(rows, cols)

def parse_vector(s, n):
    if s.startswith("Vector"):
        parts = s.split(":")
        vals = parts[2].split(",")
    else:
        vals = s.split(",")
    return np.array([float(v) for v in vals])

def parse_calib_ini(path):
    # Return dict with K (3x3), R (3x3), t (3,), width, height.
    cfg = configparser.ConfigParser()
    cfg.read(path)
    sec = cfg.sections()[0]               # e.g. 'camera_0'
    K = parse_matrix(cfg[sec]["k_matrix"], 3, 3)
    R = parse_matrix(cfg[sec]["r_matrix"], 3, 3)
    t = parse_vector(cfg[sec]["t_vector"], 3)
    w = int(cfg[sec]["width"])
    h = int(cfg[sec]["height"])
    return {"K": K, "R": R, "t": t, "width": w, "height": h}

def rotmat2qvec(R):
    # Convert 3x3 rotation matrix to quaternion (w, x, y, z).
    Rxx, Ryx, Rzx = R[0, 0], R[1, 0], R[2, 0]
    Rxy, Ryy, Rzy = R[0, 1], R[1, 1], R[2, 1]
    Rxz, Ryz, Rzz = R[0, 2], R[1, 2], R[2, 2]
    K_mat = np.array([
        [Rxx - Ryy - Rzz, 0, 0, 0],
        [Ryx + Rxy, Ryy - Rxx - Rzz, 0, 0],
        [Rzx + Rxz, Rzy + Ryz, Rzz - Rxx - Ryy, 0],
        [Ryz - Rzy, Rzx - Rxz, Rxy - Ryx, Rxx + Ryy + Rzz]
    ]) / 3.0
    eigvals, eigvecs = np.linalg.eigh(K_mat)
    qvec = eigvecs[:, np.argmax(eigvals)]
    if qvec[3] < 0:
        qvec = -qvec
    return np.array([qvec[3], qvec[0], qvec[1], qvec[2]])  # w, x, y, z

# Parse all calibration files
calib_files_sorted = sorted(glob.glob(os.path.join(data_dir, "calib_*.ini")))
all_calibs = []
for cf in calib_files_sorted:
    all_calibs.append(parse_calib_ini(cf))
print(f"Parsed {len(all_calibs)} calibration files.")

# Compute camera positions in world space
# R = world-to-camera rotation, t = w2c translation
# Camera position in world: C = -R^T @ t
cam_positions = []
for c in all_calibs:
    pos = -c["R"].T @ c["t"]
    cam_positions.append(pos)
cam_positions = np.array(cam_positions)
print(f"Camera position range:")
for axis, name in enumerate(["X", "Y", "Z"]):
    print(f"  {name}: [{cam_positions[:, axis].min():.2f}, {cam_positions[:, axis].max():.2f}]")

### 3-2. Sparse view selection (farthest-point sampling)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def farthest_point_sampling(points, n_samples):
    """Select n_samples indices via farthest-point sampling."""
    n = len(points)
    selected = [np.random.randint(n)]
    dists = np.full(n, np.inf)
    for _ in range(1, n_samples):
        last = points[selected[-1]]
        d = np.linalg.norm(points - last, axis=1)
        dists = np.minimum(dists, d)
        selected.append(int(np.argmax(dists)))
    return selected

np.random.seed(42)
selected_indices = farthest_point_sampling(cam_positions, N_TRAIN_VIEWS)
print(f"Selected {N_TRAIN_VIEWS} views (indices): {selected_indices}")

# Visualise camera distribution
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(*cam_positions.T, c='blue', s=5, alpha=0.3, label='all cameras')
sel_pos = cam_positions[selected_indices]
ax.scatter(*sel_pos.T, c='red', s=100, marker='*', label='selected')
for i, idx in enumerate(selected_indices):
    ax.text(sel_pos[i, 0], sel_pos[i, 1], sel_pos[i, 2], f' v{i}', fontsize=9)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.legend()
plt.title(f"Camera distribution ({len(cam_positions)} total, {N_TRAIN_VIEWS} selected)")
plt.tight_layout()
plt.show()

### 3-3. Convert to COLMAP text format

In [ ]:
import os, shutil, numpy as np
from PIL import Image

os.makedirs(os.path.join(CONVERTED_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(CONVERTED_DIR, "sparse", "0"), exist_ok=True)

# ── Copy & rename selected images ──────────────────────────────────
rgb_files_sorted = sorted(glob.glob(os.path.join(data_dir, "rgb_*.png")))
image_names = []
for new_id, orig_idx in enumerate(selected_indices):
    src = rgb_files_sorted[orig_idx]
    dst_name = f"{new_id:04d}.png"
    shutil.copy2(src, os.path.join(CONVERTED_DIR, "images", dst_name))
    image_names.append(dst_name)
print(f"Copied {len(image_names)} images → {CONVERTED_DIR}/images/")

# ── cameras.txt  (PINHOLE) ──────────────────────────────────────────
ref_calib = all_calibs[selected_indices[0]]
fx, fy = ref_calib["K"][0, 0], ref_calib["K"][1, 1]
cx, cy = ref_calib["K"][0, 2], ref_calib["K"][1, 2]
w, h   = ref_calib["width"], ref_calib["height"]

cam_txt = os.path.join(CONVERTED_DIR, "sparse", "0", "cameras.txt")
with open(cam_txt, "w") as f:
    f.write("# Camera list with one line of data per camera:\n")
    f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
    f.write(f"# Number of cameras: 1\n")
    f.write(f"1 PINHOLE {w} {h} {fx} {fy} {cx} {cy}\n")
print(f"Wrote {cam_txt}")

# ── images.txt ───────────────────────────────────────────────────────
img_txt = os.path.join(CONVERTED_DIR, "sparse", "0", "images.txt")
with open(img_txt, "w") as f:
    f.write("# Image list with two lines of data per image:\n")
    f.write("# IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME\n")
    f.write("# POINTS2D[] as (X, Y, POINT3D_ID)\n")
    f.write(f"# Number of images: {len(selected_indices)}\n")
    for new_id, orig_idx in enumerate(selected_indices):
        cal = all_calibs[orig_idx]
        # R = R_w2c, t = t_w2c (world-to-camera extrinsic)
        qvec = rotmat2qvec(cal["R"])
        t = cal["t"]
        img_id = new_id + 1
        name = image_names[new_id]
        f.write(f"{img_id} {qvec[0]} {qvec[1]} {qvec[2]} {qvec[3]} "
                f"{t[0]} {t[1]} {t[2]} 1 {name}\n")
        f.write("\n")  # empty line for points2D
print(f"Wrote {img_txt}")

# ── points3D.txt (empty placeholder) ────────────────────────────────
pts_txt = os.path.join(CONVERTED_DIR, "sparse", "0", "points3D.txt")
with open(pts_txt, "w") as f:
    f.write("# 3D point list (empty – will be populated by dense reconstruction)\n")
print(f"Wrote {pts_txt}")

# ── pair.txt  (nearest-neighbour pairs) ─────────────────────────────
sel_positions = cam_positions[selected_indices]
pair_txt = os.path.join(CONVERTED_DIR, "pair.txt")
with open(pair_txt, "w") as f:
    n = len(selected_indices)
    f.write(f"{n}\n")
    for i in range(n):
        dists = np.linalg.norm(sel_positions - sel_positions[i], axis=1)
        dists[i] = np.inf
        nn_ids = np.argsort(dists)[:N_SOURCE_PAIRS]
        f.write(f"{i}\n")
        f.write(f"{N_SOURCE_PAIRS}")
        for j in nn_ids:
            score = 1.0 / (dists[j] + 1e-8)
            f.write(f" {j} {score:.6f}")
        f.write("\n")
print(f"Wrote {pair_txt}")

print("\nCOLMAP text conversion complete!")

## 4. Monocular Depth Estimation (Marigold)

### 4-1. Install & run Marigold

In [ ]:
import os
import torch
import numpy as np
from PIL import Image

# Install diffusers (Marigold is built-in since v0.28)
!pip install -q diffusers accelerate

from diffusers import MarigoldDepthPipeline

os.makedirs(os.path.join(CONVERTED_DIR, "depth_npy"), exist_ok=True)

pipe = MarigoldDepthPipeline.from_pretrained(
    "prs-eth/marigold-depth-lcm-v1-0",
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")

images_dir = os.path.join(CONVERTED_DIR, "images")
depth_dir  = os.path.join(CONVERTED_DIR, "depth_npy")

image_files = sorted([f for f in os.listdir(images_dir) if f.endswith(".png")])
print(f"Running Marigold on {len(image_files)} images ...")

for fname in image_files:
    img = Image.open(os.path.join(images_dir, fname)).convert("RGB")
    output = pipe(img, num_inference_steps=4, ensemble_size=5)
    depth = np.squeeze(np.array(output.prediction))  # H x W
    stem = fname.replace(".png", "")
    out_path = os.path.join(depth_dir, f"{stem}_pred.npy")
    np.save(out_path, depth)
    print(f"  {fname} -> {stem}_pred.npy  shape={depth.shape}")

del pipe
torch.cuda.empty_cache()
print("Marigold depth estimation done.")

### 4-2. Verify & visualise depth maps

In [ ]:
import matplotlib.pyplot as plt, numpy as np, os

depth_files = sorted([f for f in os.listdir(depth_dir) if f.endswith(".npy")])
n = len(depth_files)
fig, axes = plt.subplots(n, 2, figsize=(10, 4*n))
if n == 1:
    axes = axes[None, :]

for i, df in enumerate(depth_files):
    depth = np.load(os.path.join(depth_dir, df))
    stem = df.replace("_pred.npy", ".png")
    img = np.array(Image.open(os.path.join(images_dir, stem)))

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(stem, fontsize=10)
    axes[i, 0].axis("off")

    im = axes[i, 1].imshow(depth, cmap="turbo")
    axes[i, 1].set_title(f"depth {depth.shape}", fontsize=10)
    axes[i, 1].axis("off")
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046)

    # Resolution check
    assert depth.shape[0] == img.shape[0] and depth.shape[1] == img.shape[1], \
        f"Resolution mismatch! depth={depth.shape} vs img={img.shape[:2]}"

plt.tight_layout()
plt.show()
print("All depth maps match image resolution.")

## 5. Generate Initial Point Cloud

### 5-1. Generate initial point cloud via COLMAP triangulation

Uses known camera poses + SIFT feature matching to triangulate 3D points in correct world coordinates.

In [ ]:
import os, shutil, subprocess
import numpy as np
from plyfile import PlyData, PlyElement

# ── Install COLMAP ──
print("Installing COLMAP...")
ret = subprocess.run(["apt", "install", "-y", "colmap"], capture_output=True, text=True)
if ret.returncode != 0:
    print("apt install failed, trying conda...")
    !conda install -y -c conda-forge colmap 2>/dev/null || echo "conda also failed"
else:
    print("COLMAP installed.")

# ── Paths ──
images_dir = os.path.join(CONVERTED_DIR, "images")
sparse_in  = os.path.join(CONVERTED_DIR, "sparse", "0")
sparse_out = os.path.join(CONVERTED_DIR, "sparse", "triangulated")
db_path    = os.path.join(CONVERTED_DIR, "colmap.db")
dense_dir  = os.path.join(CONVERTED_DIR, "dense")
fused_path = os.path.join(dense_dir, "fused.ply")

# Clean previous runs
if os.path.exists(db_path):
    os.remove(db_path)
if os.path.exists(fused_path):
    os.remove(fused_path)
os.makedirs(sparse_out, exist_ok=True)
os.makedirs(dense_dir, exist_ok=True)

n_images = len([f for f in os.listdir(images_dir) if f.endswith(".png")])
print(f"\nTriangulating from {n_images} images...")

# ── 1. Feature extraction ──
print("\n[1/3] Extracting SIFT features...")
!colmap feature_extractor \
    --database_path {db_path} \
    --image_path {images_dir} \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 1 2>&1 | tail -5

# ── 2. Feature matching ──
print("\n[2/3] Matching features...")
!colmap exhaustive_matcher \
    --database_path {db_path} \
    --SiftMatching.use_gpu 1 2>&1 | tail -5

# ── 3. Triangulate with known poses ──
print("\n[3/3] Triangulating with known camera poses...")
!colmap point_triangulator \
    --database_path {db_path} \
    --image_path {images_dir} \
    --input_path {sparse_in} \
    --output_path {sparse_out} 2>&1 | tail -5

# ── 4. Read triangulated points and save as fused.ply ──
print("\nReading triangulated points...")
import sys
sys.path.insert(0, REPO_DIR)
from scene.colmap_loader import read_points3D_binary, read_points3D_text

pts3d_bin = os.path.join(sparse_out, "points3D.bin")
pts3d_txt = os.path.join(sparse_out, "points3D.txt")

try:
    xyz, rgb, _ = read_points3D_binary(pts3d_bin)
except:
    try:
        xyz, rgb, _ = read_points3D_text(pts3d_txt)
    except:
        print("ERROR: No triangulated points found. COLMAP may have failed.")
        print("Check the output above for errors.")
        raise

print(f"Triangulated {len(xyz)} points")
print(f"  Center: [{xyz.mean(0)[0]:.1f}, {xyz.mean(0)[1]:.1f}, {xyz.mean(0)[2]:.1f}]")
print(f"  Spread: [{xyz.std(0)[0]:.1f}, {xyz.std(0)[1]:.1f}, {xyz.std(0)[2]:.1f}]")
print(f"  Dist from origin: mean={np.linalg.norm(xyz, axis=1).mean():.1f}")

# Save as dense/fused.ply
normals = np.zeros_like(xyz)
dtype = [('x','f4'),('y','f4'),('z','f4'),
         ('nx','f4'),('ny','f4'),('nz','f4'),
         ('red','u1'),('green','u1'),('blue','u1')]
rgb_u8 = rgb.astype(np.uint8)
elements = np.empty(len(xyz), dtype=dtype)
elements[:] = list(map(tuple, np.hstack([xyz.astype(np.float32), normals.astype(np.float32), rgb_u8])))
vertex = PlyElement.describe(elements, 'vertex')
PlyData([vertex]).write(fused_path)

print(f"\nSaved {fused_path} ({len(xyz)} points, {os.path.getsize(fused_path)/1e6:.1f} MB)")

## 6. Dataset Validation

### 6-1. Check all required files

In [ ]:
import os

required = [
    ("images/",           True),
    ("sparse/0/cameras.txt", False),
    ("sparse/0/images.txt",  False),
    ("sparse/0/points3D.txt",False),
    ("pair.txt",          False),
    ("dense/fused.ply",   False),
    ("depth_npy/",        True),
]

all_ok = True
for rel, is_dir in required:
    full = os.path.join(CONVERTED_DIR, rel)
    exists = os.path.isdir(full) if is_dir else os.path.isfile(full)
    status = "OK" if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"  [{status}]  {rel}")

# Count files
n_depth = len([f for f in os.listdir(os.path.join(CONVERTED_DIR, "depth_npy"))
               if f.endswith(".npy")])
n_img = len([f for f in os.listdir(os.path.join(CONVERTED_DIR, "images"))
             if f.endswith(".png")])
print(f"\n  Images: {n_img},  Depth maps: {n_depth}")
assert n_depth == n_img, "Mismatch between images and depth maps!"

assert all_ok, "Some required files are missing!"
print("\nDataset validation PASSED.")

## 7. Patch FatesGS for Custom Data

### 7-1. Patch `dataset_readers.py` — auto-detect image resolution

In [ ]:
import os

dr_path = os.path.join(REPO_DIR, "scene", "dataset_readers.py")
with open(dr_path, "r") as f:
    src = f.read()

# Back up
backup_dr = dr_path + ".bak"
if not os.path.exists(backup_dr):
    with open(backup_dr, "w") as f:
        f.write(src)
    print(f"Backup saved: {backup_dr}")

# Replace the hardcoded resolution block
old_block = """\
    if args.resolution == 2:
        ori_w, ori_h = 1600, 1200
    else:
        ori_w, ori_h = 768, 576"""

new_block = """\
    # --- PATCHED: auto-detect resolution from first image ---
    _first_img = Image.open(image_paths[0])
    ori_w, ori_h = _first_img.size  # (width, height)
    _first_img.close()
    print(f"[PATCH] Auto-detected image size: {ori_w}x{ori_h}")
    # --- END PATCH ---"""

if old_block in src:
    src = src.replace(old_block, new_block)
    with open(dr_path, "w") as f:
        f.write(src)
    print("dataset_readers.py patched successfully.")
else:
    if "PATCHED: auto-detect" in src:
        print("dataset_readers.py already patched.")
    else:
        print("WARNING: Could not find expected code block to patch!")

### 7-2. Patch `loss_utils.py` — fix CenterCrop sizes for 800x800

In [ ]:
import os

lu_path = os.path.join(REPO_DIR, "utils", "loss_utils.py")
with open(lu_path, "r") as f:
    src = f.read()

backup_lu = lu_path + ".bak"
if not os.path.exists(backup_lu):
    with open(backup_lu, "w") as f:
        f.write(src)
    print(f"Backup saved: {backup_lu}")

# Original DTU crops: (576, 768) and (544, 736)
# For 800x800 at RESOLUTION=1: (768, 768) and (736, 736)
# For 800x800 at RESOLUTION=2: (384, 384) and (352, 352)

if RESOLUTION == 1:
    new_crop1, new_crop2 = "(768, 768)", "(736, 736)"
else:
    new_crop1, new_crop2 = "(384, 384)", "(352, 352)"

patched = False

# Try original DTU values first
if "CenterCrop((576, 768))" in src:
    src = src.replace("CenterCrop((576, 768))", f"CenterCrop({new_crop1})")
    src = src.replace("CenterCrop((544, 736))", f"CenterCrop({new_crop2})")
    patched = True

if patched:
    with open(lu_path, "w") as f:
        f.write(src)
    print(f"loss_utils.py patched: crops -> {new_crop1}, {new_crop2}")
elif new_crop1.strip("()") in src:
    print("loss_utils.py already patched.")
else:
    print("WARNING: Could not find CenterCrop lines to patch!")

## 8. Training

### 8-1. Run train.py

In [ ]:
%cd {REPO_DIR}

!python train.py \
    -s {CONVERTED_DIR} \
    -m {OUTPUT_DIR} \
    -r {RESOLUTION} \
    --iterations {ITERATIONS} \
    --lambda_dssim {LAMBDA_DSSIM} \
    --lambda_dist {LAMBDA_DIST} \
    --lambda_normal {LAMBDA_NORMAL} \
    --lambda_feat {LAMBDA_FEAT} \
    --lambda_depth {LAMBDA_DEPTH} \
    --save_iterations {ITERATIONS} \
    --test_iterations {ITERATIONS}

### 8-2. Check training output

In [ ]:
import os, glob

ckpt_dir = os.path.join(OUTPUT_DIR, "point_cloud")
if os.path.isdir(ckpt_dir):
    iters = sorted(os.listdir(ckpt_dir))
    print(f"Saved checkpoints: {iters}")
    for it in iters:
        files = os.listdir(os.path.join(ckpt_dir, it))
        print(f"  {it}/: {files}")
else:
    print("No point_cloud directory found — training may have failed.")

## 9. Mesh Extraction

### 9-1. Run render.py (mesh extraction)

In [ ]:
%cd {REPO_DIR}

!python render.py \
    -s {CONVERTED_DIR} \
    -m {OUTPUT_DIR} \
    -r {RESOLUTION} \
    --skip_test \
    --skip_train \
    --mesh_res {MESH_RES} \
    --num_cluster {NUM_CLUSTER}

### 9-2. Check mesh files

In [ ]:
import os, glob

train_dir = os.path.join(OUTPUT_DIR, "train")
if os.path.isdir(train_dir):
    for root, dirs, files in os.walk(train_dir):
        for f in files:
            fp = os.path.join(root, f)
            size_mb = os.path.getsize(fp) / 1e6
            print(f"  {os.path.relpath(fp, OUTPUT_DIR):50s}  {size_mb:.1f} MB")
else:
    print("No train/ directory found.")

## 10. Visualization

### 10-1. Rendered images vs Ground Truth

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Render train images if not already done
train_render_dir = None
train_dir_base = os.path.join(OUTPUT_DIR, "train")
if os.path.isdir(train_dir_base):
    subdirs = sorted(os.listdir(train_dir_base))
    for sd in subdirs:
        renders_dir = os.path.join(train_dir_base, sd, "renders")
        if os.path.isdir(renders_dir):
            train_render_dir = renders_dir
            break

if train_render_dir is None:
    print("Running render.py --skip_test --skip_mesh to get rendered training images ...")
    %cd {REPO_DIR}
    !python render.py -s {CONVERTED_DIR} -m {OUTPUT_DIR} -r {RESOLUTION} --skip_test --skip_mesh
    # Find again
    for sd in sorted(os.listdir(train_dir_base)):
        renders_dir = os.path.join(train_dir_base, sd, "renders")
        if os.path.isdir(renders_dir):
            train_render_dir = renders_dir
            break

if train_render_dir:
    gt_dir = train_render_dir.replace("renders", "gt")
    render_files = sorted(glob.glob(os.path.join(train_render_dir, "*.png")))
    n = len(render_files)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4*n))
    if n == 1:
        axes = axes[None, :]
    for i, rf in enumerate(render_files):
        fname = os.path.basename(rf)
        rend = np.array(Image.open(rf))
        axes[i, 0].imshow(rend)
        axes[i, 0].set_title(f"Rendered: {fname}", fontsize=10)
        axes[i, 0].axis("off")
        gt_path = os.path.join(gt_dir, fname)
        if os.path.exists(gt_path):
            gt = np.array(Image.open(gt_path))
            axes[i, 1].imshow(gt)
            axes[i, 1].set_title(f"GT: {fname}", fontsize=10)
        else:
            axes[i, 1].text(0.5, 0.5, "GT not found", ha='center', va='center')
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Could not find rendered images.")

### 10-2. Depth maps visualization

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

depth_render_dir = None
train_dir_base = os.path.join(OUTPUT_DIR, "train")
if os.path.isdir(train_dir_base):
    for sd in sorted(os.listdir(train_dir_base)):
        d_dir = os.path.join(train_dir_base, sd, "vis")
        if os.path.isdir(d_dir):
            depth_render_dir = d_dir
            break

if depth_render_dir:
    depth_files = sorted(glob.glob(os.path.join(depth_render_dir, "depth_*.tiff")))
    n = len(depth_files)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    if n == 1:
        axes = [axes]
    for i, df in enumerate(depth_files):
        d = np.array(Image.open(df))
        axes[i].imshow(d, cmap="turbo")
        axes[i].set_title(os.path.basename(df), fontsize=10)
        axes[i].axis("off")
    plt.suptitle("Rendered Depth Maps", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No rendered depth directory found. Re-run render.py with depth output.")

### 10-3. 3D Mesh with Plotly

In [ ]:
import os, glob
import plotly.graph_objects as go
import trimesh

# Find the post-processed mesh
mesh_path = None
train_dir_base = os.path.join(OUTPUT_DIR, "train")
if os.path.isdir(train_dir_base):
    for root, dirs, files in os.walk(train_dir_base):
        for f in files:
            if f.endswith("_post.ply"):
                mesh_path = os.path.join(root, f)
                break

if mesh_path is None:
    # Fallback: look for fuse.ply
    for root, dirs, files in os.walk(train_dir_base):
        for f in files:
            if f == "fuse.ply":
                mesh_path = os.path.join(root, f)
                break

if mesh_path:
    print(f"Loading mesh: {mesh_path}")
    mesh = trimesh.load(mesh_path)
    verts = mesh.vertices
    faces = mesh.faces

    fig = go.Figure(data=[
        go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
            vertexcolor=mesh.visual.vertex_colors[:, :3] if hasattr(mesh.visual, 'vertex_colors') else None,
            opacity=1.0,
        )
    ])
    fig.update_layout(
        title="Reconstructed Mesh",
        scene=dict(aspectmode='data'),
        width=800, height=600,
    )
    fig.show()
else:
    print("No mesh file found. Run mesh extraction first.")

## 11. Utilities

### 11-1. Download results as ZIP

In [ ]:
import shutil
from google.colab import files

zip_name = f"/content/{SCENE_NAME}_results"
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f"Created {zip_name}.zip")
files.download(f"{zip_name}.zip")

### 11-2. GPU memory check

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    mem = torch.cuda.mem_get_info()
    print(f"Free: {mem[0]/1e9:.2f} GB / Total: {mem[1]/1e9:.2f} GB")

### 11-3. Clear GPU cache

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("GPU cache cleared.")
if torch.cuda.is_available():
    mem = torch.cuda.mem_get_info()
    print(f"Free: {mem[0]/1e9:.2f} GB / Total: {mem[1]/1e9:.2f} GB")

### 11-4. Restore original code (undo patches)

In [ ]:
import os, shutil

for path in [
    os.path.join(REPO_DIR, "scene", "dataset_readers.py"),
    os.path.join(REPO_DIR, "utils", "loss_utils.py"),
]:
    bak = path + ".bak"
    if os.path.exists(bak):
        shutil.copy2(bak, path)
        print(f"Restored: {path}")
    else:
        print(f"No backup found for: {path}")

print("Patches reverted.")

---
## Diagnostics

### D-1. fused.ply Point Cloud Verification

In [ ]:
import numpy as np, os
from plyfile import PlyData
import matplotlib.pyplot as plt

ply_path = os.path.join(CONVERTED_DIR, "dense", "fused.ply")
plydata = PlyData.read(ply_path)
pts = np.stack([plydata['vertex']['x'], plydata['vertex']['y'], plydata['vertex']['z']], axis=1)
dists = np.linalg.norm(pts, axis=1)

print(f"Total points: {len(pts)}")
print(f"Distance from origin:")
print(f"  mean:   {dists.mean():.1f}")
print(f"  median: {np.median(dists):.1f}")
print(f"  min:    {dists.min():.1f}")
print(f"  max:    {dists.max():.1f}")
print(f"Bounding box:")
print(f"  min: [{pts.min(0)[0]:.1f}, {pts.min(0)[1]:.1f}, {pts.min(0)[2]:.1f}]")
print(f"  max: [{pts.max(0)[0]:.1f}, {pts.max(0)[1]:.1f}, {pts.max(0)[2]:.1f}]")

# Verdict
if 200 < dists.mean() < 600:
    print(f"\nVERDICT: Point cloud scale looks REASONABLE (object near origin, cameras at ~400)")
else:
    print(f"\nVERDICT: Point cloud scale looks WRONG (expected mean dist ~300-500, got {dists.mean():.1f})")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].hist(dists, bins=100, color='steelblue')
axes[0].axvline(400, color='red', linestyle='--', label='camera dist (400)')
axes[0].set_title('Distance from origin'); axes[0].legend()

axes[1].scatter(pts[::10, 0], pts[::10, 1], s=0.5, alpha=0.3)
# Plot camera positions
for idx in selected_indices:
    cp = -all_calibs[idx]["R"].T @ all_calibs[idx]["t"]
    axes[1].plot(cp[0], cp[1], 'r*', markersize=10)
axes[1].set_title('Point cloud XY (red=cameras)'); axes[1].set_aspect('equal')

axes[2].scatter(pts[::10, 0], pts[::10, 2], s=0.5, alpha=0.3)
for idx in selected_indices:
    cp = -all_calibs[idx]["R"].T @ all_calibs[idx]["t"]
    axes[2].plot(cp[0], cp[2], 'r*', markersize=10)
axes[2].set_title('Point cloud XZ (red=cameras)'); axes[2].set_aspect('equal')

plt.tight_layout(); plt.show()

### D-2. Marigold Depth Quality

In [ ]:
import numpy as np, matplotlib.pyplot as plt, os
from PIL import Image

images_dir = os.path.join(CONVERTED_DIR, "images")
depth_dir = os.path.join(CONVERTED_DIR, "depth_npy")

n_imgs = len([f for f in os.listdir(images_dir) if f.endswith(".png")])
fig, axes = plt.subplots(n_imgs, 3, figsize=(12, 4*n_imgs))
if n_imgs == 1: axes = axes[None, :]

for i in range(n_imgs):
    img = np.array(Image.open(os.path.join(images_dir, f"{i:04d}.png")))
    depth = np.load(os.path.join(depth_dir, f"{i:04d}_pred.npy"))

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"RGB {i}"); axes[i, 0].axis("off")

    im = axes[i, 1].imshow(depth, cmap="turbo")
    axes[i, 1].set_title(f"Depth {i} ({depth.min():.3f}~{depth.max():.3f})")
    axes[i, 1].axis("off"); plt.colorbar(im, ax=axes[i, 1], fraction=0.046)

    dy = np.abs(depth[1:, :] - depth[:-1, :])
    dx = np.abs(depth[:, 1:] - depth[:, :-1])
    axes[i, 2].imshow(dy[:, :-1] + dx[:-1, :], cmap="hot")
    axes[i, 2].set_title(f"Depth edges {i}"); axes[i, 2].axis("off")

plt.suptitle("Marigold Depth Quality", fontsize=14)
plt.tight_layout(); plt.show()

### D-3. VisMVSNet Feature Quality

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, os, sys
from PIL import Image
from sklearn.decomposition import PCA

sys.path.insert(0, REPO_DIR)
from utils.feat_utils import FeatExt
from utils.general_utils import PILtoTorch

images_dir = os.path.join(CONVERTED_DIR, "images")
feat_ext = FeatExt().cuda(); feat_ext.eval()
mean = torch.tensor([0.485, 0.456, 0.406]).float()
std = torch.tensor([0.229, 0.224, 0.225]).float()

n_imgs = len([f for f in os.listdir(images_dir) if f.endswith(".png")])
fig, axes = plt.subplots(n_imgs, 3, figsize=(12, 4*n_imgs))
if n_imgs == 1: axes = axes[None, :]

for i in range(n_imgs):
    img_pil = Image.open(os.path.join(images_dir, f"{i:04d}.png"))
    img_tensor = torch.cat([PILtoTorch(im, img_pil.size) for im in img_pil.split()[:3]], dim=0)
    img_norm = (img_tensor.unsqueeze(0) / 2 + 0.5 - mean.view(1,3,1,1)) / std.view(1,3,1,1)
    with torch.no_grad():
        f1, f2, f3 = feat_ext(img_norm.cuda())

    axes[i, 0].imshow(np.array(img_pil))
    axes[i, 0].set_title(f"RGB {i}"); axes[i, 0].axis("off")

    feat_vis = f3[0].cpu().numpy()
    feat_mag = np.linalg.norm(feat_vis, axis=0)
    im = axes[i, 1].imshow(feat_mag, cmap="viridis")
    axes[i, 1].set_title(f"Feature mag {feat_vis.shape}"); axes[i, 1].axis("off")
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046)

    C, H, W = feat_vis.shape
    pca = PCA(n_components=3)
    feat_pca = pca.fit_transform(feat_vis.reshape(C, -1).T).reshape(H, W, 3)
    feat_pca = (feat_pca - feat_pca.min()) / (feat_pca.max() - feat_pca.min() + 1e-8)
    axes[i, 2].imshow(feat_pca)
    axes[i, 2].set_title(f"Feature PCA"); axes[i, 2].axis("off")

plt.suptitle("VisMVSNet Feature Quality", fontsize=14)
plt.tight_layout(); plt.show()
del feat_ext; torch.cuda.empty_cache()

### D-4. Depth Scaling: median vs percentile vs center

In [ ]:
import numpy as np, os
from PIL import Image
from skimage.filters import threshold_otsu

images_dir = os.path.join(CONVERTED_DIR, "images")
depth_dir = os.path.join(CONVERTED_DIR, "depth_npy")
n_imgs = len([f for f in os.listdir(images_dir) if f.endswith(".png")])

print("="*90)
print(f"{'View':>5} | {'median':>8} | {'otsu_th':>8} | {'fg_med':>8} | {'obj@med':>9} | {'obj@otsu':>9} | {'actual':>8}")
print("="*90)

for i in range(n_imgs):
    cal = all_calibs[selected_indices[i]]
    cam_pos = -cal["R"].T @ cal["t"]
    expected = np.linalg.norm(cam_pos)

    depth = np.load(os.path.join(depth_dir, f"{i:04d}_pred.npy"))
    h, w = depth.shape
    ctr = depth[h//2, w//2]

    med = np.median(depth)
    valid = depth[depth > 1e-6]
    thresh = threshold_otsu(valid)
    fg = valid[valid < thresh]
    fg_med = np.median(fg) if len(fg) > 100 else np.median(valid)

    obj_med = ctr / (med + 1e-8) * expected if ctr > 1e-6 else 0
    obj_otsu = ctr / (fg_med + 1e-8) * expected if ctr > 1e-6 else 0

    print(f"{i:>5} | {med:>8.4f} | {thresh:>8.4f} | {fg_med:>8.4f} | {obj_med:>9.1f} | {obj_otsu:>9.1f} | {expected:>8.1f}")

print()
print("otsu_th:  Otsu threshold (separates object from background)")
print("fg_med:   median of foreground (depth < otsu_th) — used for scaling")
print("obj@med:  object distance with global median (OLD)")
print("obj@otsu: object distance with Otsu foreground median (CURRENT FIX)")
print("actual:   real distance (400)")